# Фильтрация сигнала с помощью FFT

## Теоретическое обоснование

В этом ноутбуке мы рассмотрим метод фильтрации сигнала с использованием Быстрого Преобразования Фурье (FFT).

### Дискретное преобразование Фурье

Дискретное преобразование Фурье (DFT) определяется формулой:

$$X_k = \sum_{n=0}^{N-1} x_n \cdot e^{-i 2\pi k n / N}$$

где:
- $x_n$ - входной сигнал
- $N$ - количество отсчетов
- $X_k$ - комплексные коэффициенты Фурье

Обратное преобразование Фурье:

$$x_n = \frac{1}{N} \sum_{k=0}^{N-1} X_k \cdot e^{i 2\pi k n / N}$$

### Генерация сигнала

Мы создаем сигнал, состоящий из двух гармонических компонент:

$$y(t) = 2\cdot\sin(2\pi f_1 t + 0.2) + 3\cdot\cos(2\pi f_2 t + 0.3) + \text{noise}$$

где:
- $f_1 = 20$ Hz
- $f_2 = 30$ Hz
- noise - случайный шум

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy as sc

plt.style.use('ggplot')
np.random.seed(20)

### Настройка параметров сигнала

In [ ]:
# Время
t = np.linspace(0, 1, 1000)

# Частоты в сигнале
f1 = 20
f2 = 30

# Случайный шум для добавления к сигналу
noise = np.random.random_sample(len(t))

# Полный сигнал
y = 2*np.sin(2*np.pi*f1*t+0.2) + 3*np.cos(2*np.pi*f2*t+0.3) + noise*5

# Часть сигнала, которую мы хотим выделить
y1 = 2*np.sin(2*np.pi*f1*t+0.2)

### Вычисление FFT

Применяем быстрое преобразование Фурье к сигналу:

$$F = \mathcal{F}\{y(t)\}$$

In [ ]:
# FFT сигнала
F = sc.fft.fft(y)

# Другие параметры
N = len(t)  # количество отсчетов
dt = 0.001  # разница во времени между отсчетами
w = np.fft.fftfreq(N, dt)  # список частот для FFT
pFrequency = np.where(w >= 0)[0]  # только положительные частоты
magnitudeF = abs(F[:len(pFrequency)])  # амплитуда F для положительных частот

### Функция для фильтрации частот

Фильтрация осуществляется путем обнуления коэффициентов Фурье вне заданного диапазона частот:

$$F_{\text{filtered}}[k] = \begin{cases} 
F[k] & \text{если } f_{\min} \leq f_k \leq f_{\max} \\
0 & \text{иначе}
\end{cases}$$

In [ ]:
def blockHigherFreq(FT, fmin, fmax, plot=False):
    """
    Функция фильтрации: блокирует частоты выше fmax и ниже fmin
    и возвращает очищенный FT
    """
    FT_filtered = FT.copy()
    for i in range(len(FT_filtered)):
        freq = abs(w[i])  # Абсолютное значение частоты
        if (freq >= fmax) or (freq <= fmin):
            FT_filtered[i] = 0
    
    if plot:
        plt.plot(pFrequency, abs(FT_filtered[:len(pFrequency)]))
        plt.xlabel('Гц')
        plt.ylabel('Амплитуда')
        plt.title('Очищенный FFT')
        plt.grid(True)
        plt.show()
    
    return FT_filtered

def normalise(signal):
    """Функция нормализации (приводит сигнал к масштабу от 0 до 1)"""
    M = max(signal)
    normalised = signal/M
    return normalised

def pltfft():
    """Построение графика FFT"""
    plt.plot(pFrequency, magnitudeF)
    plt.xlabel('Гц')
    plt.ylabel('Амплитуда')
    plt.title('FFT полного сигнала')
    plt.grid(True)
    plt.show()

def pltCompleteSignal():
    """Построение графика полного сигнала"""
    plt.plot(t, y, 'b')
    plt.xlabel('Время (с)')
    plt.ylabel('Амплитуда')
    plt.title('Полный сигнал')
    plt.grid(True)
    plt.show()

### Обратное преобразование Фурье

После фильтрации применяем обратное преобразование Фурье для восстановления сигнала:

$$y_{\text{filtered}}(t) = \mathcal{F}^{-1}\{F_{\text{filtered}}\}$$

In [ ]:
# Очистка FT путем выбора только частот между 18 и 22 Гц
newFT = blockHigherFreq(F, 18, 22, plot=True)

# Восстановление очищенного сигнала с помощью обратного FFT
cleanedSignal = sc.fft.ifft(newFT).real  # Берем вещественную часть для удаления малых мнимых компонентов

# Расчет ошибки
error = normalise(y1) - normalise(cleanedSignal)

### Визуализация результатов

Сравнение исходного сигнала, отфильтрованного сигнала и ошибки:

In [ ]:
# Построение графиков результатов
pltCompleteSignal()  # График полного сигнала
pltfft()  # График FFT

plt.figure(figsize=(10, 8))

# Subplot 1: Исходный сигнал
plt.subplot(3, 1, 1)
plt.title('Исходный сигнал с шумом')
plt.plot(t, y, 'g', alpha=0.7)
plt.grid(True)

# Subplot 2: Сравнение очищенного сигнала
plt.subplot(3, 1, 2)
plt.plot(t, normalise(cleanedSignal), label='Очищенный сигнал', color='b')
plt.plot(t, normalise(y1), label='Целевой сигнал', ls='-', color='r')
plt.title('Очищенный сигнал и целевой сигнал')
plt.legend()
plt.grid(True)

# Subplot 3: Ошибка
plt.subplot(3, 1, 3)
plt.plot(t, error, color='r', label='Ошибка')
plt.title('Ошибка между очищенным и целевым сигналом')
plt.xlabel('Время (с)')
plt.grid(True)

plt.tight_layout()
plt.show()

### Анализ результатов

Как видно из графиков, метод FFT-фильтрации успешно выделяет компоненту 20 Hz из зашумленного сигнала. Ошибка восстановления минимальна, что демонстрирует эффективность данного подхода для частотной фильтрации сигналов.